# Dependencies

In [67]:
! pip install torch torchvision torchaudio scikit-learn pandas numpy tqdm


In [68]:

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import f1_score, classification_report, matthews_corrcoef, balanced_accuracy_score

# Hyperparameters

In [69]:
SEQ_LEN = 48
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

stock_path = "../data/meta_stock.csv"
news_path  = "../data/Meta_news_embeddings.csv"

# Data Merging

In [70]:
stock_df = pd.read_csv(stock_path, parse_dates=["timestamp"])
news_df  = pd.read_csv(news_path, parse_dates=["date"])
stock_df["date"] = stock_df["timestamp"].dt.date
news_df["date"]  = news_df["date"].dt.date

merged = pd.merge(stock_df, news_df, on=["ticker", "date"], how="left")

emb_cols = [c for c in merged.columns if c.startswith("emb_")]
for c in emb_cols:
    merged[c] = merged[c].fillna(0.0)
merged = merged.dropna(subset=["target_up_bin"])
merged = merged.sort_values("timestamp")

exclude_cols = [
    "timestamp", "date", "ticker",
    "target_5m_return", "target_up_bin",
    "title", "link", "media"
]
feature_cols = [c for c in merged.columns if c not in exclude_cols]

merged[feature_cols] = merged[feature_cols].fillna(0).replace([np.inf, -np.inf], 0)

scaler = StandardScaler()
merged[feature_cols] = scaler.fit_transform(merged[feature_cols])

In [71]:
merged.head()

,timestamp,ticker,open,high,low,close,volume,return_5m,return_30m,return_1h,...,emb_374,emb_375,emb_376,emb_377,emb_378,emb_379,emb_380,emb_381,emb_382,emb_383
0,2025-07-15 08:00:00+00:00,META,-0.594914,-0.609082,-0.592229,-0.607182,-0.452397,-0.012701,-0.029075,-0.042169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-07-15 08:05:00+00:00,META,-0.614906,-0.629033,-0.599935,-0.614868,-0.467823,-0.199223,-0.029075,-0.042169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2025-07-15 08:10:00+00:00,META,-0.605064,-0.619211,-0.603326,-0.608104,-0.458288,0.151495,-0.029075,-0.042169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2025-07-15 08:15:00+00:00,META,-0.601681,-0.611845,-0.586681,-0.597651,-0.454780,0.240980,-0.029075,-0.042169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-07-15 08:20:00+00:00,META,-0.580150,-0.594350,-0.565105,-0.580127,-0.467595,0.412387,-0.029075,-0.042169,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [72]:
class StockNewsDataset(Dataset):
    def __init__(self, df, seq_len=24):
        self.df = df
        self.seq_len = seq_len
        self.X = df[feature_cols].values
        self.y = df["target_up_bin"].astype(int).values
    def __len__(self):
        return len(self.df) - self.seq_len
    def __getitem__(self, idx):
        x_seq = self.X[idx:idx+self.seq_len]
        y = self.y[idx+self.seq_len]
        return torch.tensor(x_seq, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

dataset = StockNewsDataset(merged, SEQ_LEN)
train_size = int(len(dataset)*0.8)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# Train

In [73]:
class StockNewsLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        h = h[-1]
        return self.fc(h)

model = StockNewsLSTM(input_dim=len(feature_cols)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [74]:
print(f"🚀 Training on {DEVICE} ...")
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for xb, yb in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}", ncols=100):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            out = model(xb)
            preds += out.argmax(1).cpu().tolist()
            labels += yb.cpu().tolist()
    
    f1 = f1_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    bacc = balanced_accuracy_score(labels, preds)
    direction_acc = np.mean(np.array(preds) == np.array(labels))

    print(f"Epoch {epoch+1}: loss={train_loss/len(train_dl):.4f}, " f"F1={f1:.4f}, MCC={mcc:.4f}, BAcc={bacc:.4f}, DirAcc={direction_acc:.4f}")

print("✅ Training done.")
print(classification_report(labels, preds, digits=3))

🚀 Training on cpu ...


Epoch 1/15: 100%|███████████████████████████████████████████████████| 47/47 [00:02<00:00, 21.59it/s]


Epoch 1: loss=0.5713, F1=0.0103, MCC=0.0288, BAcc=0.5017, DirAcc=0.7434


Epoch 2/15: 100%|███████████████████████████████████████████████████| 47/47 [00:02<00:00, 23.12it/s]


Epoch 2: loss=0.5428, F1=0.0305, MCC=0.0826, BAcc=0.5069, DirAcc=0.7460


...

Epoch 13: loss=0.5300, F1=0.0574, MCC=0.0400, BAcc=0.5066, DirAcc=0.7380


Epoch 14/15: 100%|██████████████████████████████████████████████████| 47/47 [00:02<00:00, 16.44it/s]


Epoch 14: loss=0.5268, F1=0.0837, MCC=0.0606, BAcc=0.5117, DirAcc=0.7380


Epoch 15/15: 100%|██████████████████████████████████████████████████| 47/47 [00:02<00:00, 17.29it/s]


Epoch 15: loss=0.5255, F1=0.0845, MCC=0.0732, BAcc=0.5135, DirAcc=0.7407
✅ Training done.
              precision    recall  f1-score   support

           0      0.749     0.980     0.849       559
           1      0.450     0.047     0.085       193

    accuracy                          0.741       752
   macro avg      0.599     0.513     0.467       752
weighted avg      0.672     0.741     0.653       752



In [75]:
import numpy as np
preds_arr = np.array(preds)
labels_arr = np.array(labels)
print("Pred 1 ratio:", (preds_arr==1).mean())
print("Label 1 ratio:", (labels_arr==1).mean())

Pred 1 ratio: 0.026595744680851064
Label 1 ratio: 0.2566489361702128
